In [83]:
import pandas as pd
import numpy as np
import os
import json

import optuna
from optuna import Trial
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import kaggle

from imblearn.over_sampling import SMOTE

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']


In [84]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [85]:
def recall_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek recall değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki recall oranı.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    P = y_true.sum()

    return float(tp_at_k / P) if P > 0 else 0.0


def lift_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek lift (precision/prevalence) değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki lift değeri.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    precision_at_k = tp_at_k / m
    prevalence = y_true.mean()

    return float(precision_at_k / prevalence) if prevalence > 0 else 0.0


def convert_auc_to_gini(auc):
    """
    ROC AUC skorunu Gini katsayısına dönüştürür.

    Gini katsayısı, ROC AUC skorunun doğrusal bir dönüşümüdür.

    Parametreler:
        auc (float): ROC AUC skoru (0 ile 1 arasında).

    Döndürür:
        float: Gini katsayısı (-1 ile 1 arasında).
    """
    return 2 * auc - 1


def ing_hubs_datathon_metric(y_true, y_prob):
    """
    Gini, recall@10% ve lift@10% metriklerini birleştiren özel bir metrik hesaplar.

    Metrik, her bir skoru bir baseline modelin metrik değerlerine göre oranlar ve aşağıdaki ağırlıkları uygular:
    - Gini: %40
    - Recall@10%: %30
    - Lift@10%: %30

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.

    Döndürür:
        float: Ağırlıklandırılmış bileşik skor.
    """
    # final metrik için ağırlıklar
    score_weights = {
        "gini": 0.4,
        "recall_at_10perc": 0.3,
        "lift_at_10perc": 0.3,
    }

    # baseline modelin her bir metrik için değerleri
    baseline_scores = {
        "roc_auc": 0.6925726757936908,
        "recall_at_10perc": 0.18469015795868773,
        "lift_at_10perc": 1.847159286784029,
    }

    # y_prob tahminleri için metriklerin hesaplanması
    roc_auc = roc_auc_score(y_true, y_prob)
    recall_at_10perc = recall_at_k(y_true, y_prob, k=0.1)
    lift_at_10perc = lift_at_k(y_true, y_prob, k=0.1)

    new_scores = {
        "roc_auc": roc_auc,
        "recall_at_10perc": recall_at_10perc,
        "lift_at_10perc": lift_at_10perc,
    }

    # roc auc değerlerinin gini değerine dönüştürülmesi
    baseline_scores["gini"] = convert_auc_to_gini(baseline_scores["roc_auc"])
    new_scores["gini"] = convert_auc_to_gini(new_scores["roc_auc"])

    # baseline modeline oranlama
    final_gini_score = new_scores["gini"] / baseline_scores["gini"]
    final_recall_score = new_scores["recall_at_10perc"] / baseline_scores["recall_at_10perc"]
    final_lift_score = new_scores["lift_at_10perc"] / baseline_scores["lift_at_10perc"]

    # ağırlıklandırılmış metriğin hesaplanması
    final_score = (
        final_gini_score * score_weights["gini"] +
        final_recall_score * score_weights["recall_at_10perc"] + 
        final_lift_score * score_weights["lift_at_10perc"]
    )
    return final_score

In [86]:
train_data

,age,tenure,cust_age_month,uses_mobile_eft,uses_cc,uses_any_digital_channel,mobile_eft_cnt_mean,mobile_eft_cnt_std,mobile_eft_cnt_min,mobile_eft_cnt_max,...,work_type_Unemployed,work_sector_Finance,work_sector_Healthcare,work_sector_Manufacturing,work_sector_Public Sector,work_sector_Retail,work_sector_Retired,work_sector_Student,work_sector_Technology,work_sector_Unemployed
0,64,135,633,1,0,1,2.238095,1.220851,1.0,5.0,...,0,0,0,0,0,0,0,0,1,0
1,22,47,217,1,1,1,1.676471,1.006662,1.0,4.0,...,0,0,0,0,0,0,0,1,0,0
2,27,108,216,1,1,1,2.555556,1.476309,1.0,6.0,...,0,1,0,0,0,0,0,0,0,0
3,40,187,293,1,1,1,7.142857,3.307839,4.0,14.0,...,1,0,0,0,0,0,0,0,0,1
4,64,218,550,1,1,1,0.793103,1.372675,0.0,5.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133282,54,217,431,1,1,1,1.393939,1.657170,0.0,6.0,...,0,0,0,0,1,0,0,0,0,0
133283,47,37,527,1,1,1,2.000000,1.174440,1.0,5.0,...,0,0,0,0,1,0,0,0,0,0
133284,66,227,565,1,1,1,9.055556,5.796277,1.0,22.0,...,0,0,0,0,0,0,1,0,0,0
133285,31,156,216,1,1,1,3.576923,1.836803,1.0,7.0,...,0,0,0,0,0,0,0,0,0,0


In [87]:
X = train_data.drop("churn", axis=1)
y = train_data["churn"]

In [88]:
def objective(trial: Trial, X: pd.DataFrame, y: np.ndarray, n_splits: int = 5) -> float:
    # Hyperparametreler
    params = {
        'objective': 'binary',
        'verbose': False,
        'metric': 'binary_logloss',
        'verbosity': -1,
        'random_state': 42,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 30, 200),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
    }

    # Dengesizlik: Pozitif sınıf ağırlığını optimize et
    neg, pos = np.bincount(y)
    scale_pos_weight = neg / pos
    params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', scale_pos_weight * 0.5, scale_pos_weight * 2)

    # K-Fold
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    custom_scores = []

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]


        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]

        # Özel metrik
        score = ing_hubs_datathon_metric(y_val, y_pred_proba)
        custom_scores.append(score)

    return np.mean(custom_scores)

In [89]:
# Optimize et
study = optuna.create_study(direction='maximize', study_name='lgbm-churn-ing-metric')
study.optimize(
    lambda trial: objective(trial, X, y),
    n_trials=100,
    show_progress_bar=True
)

print("Best trial score:", study.best_trial.value)
print("Best params:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

[I 2025-10-11 21:32:30,665] A new study created in memory with name: lgbm-churn-ing-metric
Best trial: 0. Best value: 1.03929:   1%|          | 1/100 [00:31<52:31, 31.83s/it]

[I 2025-10-11 21:33:02,498] Trial 0 finished with value: 1.0392900040656396 and parameters: {'num_leaves': 197, 'max_depth': 9, 'learning_rate': 0.10265816227737637, 'n_estimators': 240, 'subsample': 0.9668049046880721, 'colsample_bytree': 0.828840476152519, 'reg_alpha': 7.651274207381662, 'reg_lambda': 6.078469639888152, 'min_child_samples': 90, 'subsample_freq': 6, 'scale_pos_weight': 8.149988533816416}. Best is trial 0 with value: 1.0392900040656396.


Best trial: 1. Best value: 1.04825:   2%|▏         | 2/100 [01:12<1:00:29, 37.04s/it]

[I 2025-10-11 21:33:43,183] Trial 1 finished with value: 1.0482510418388444 and parameters: {'num_leaves': 45, 'max_depth': 7, 'learning_rate': 0.050790475336444124, 'n_estimators': 574, 'subsample': 0.8491092757668167, 'colsample_bytree': 0.8618655873013453, 'reg_alpha': 5.2665492591409135, 'reg_lambda': 0.9660819950141175, 'min_child_samples': 12, 'subsample_freq': 9, 'scale_pos_weight': 9.826665925508141}. Best is trial 1 with value: 1.0482510418388444.


Best trial: 1. Best value: 1.04825:   3%|▎         | 3/100 [02:34<1:32:48, 57.41s/it]

...

[I 2025-10-11 23:33:58,633] Trial 97 finished with value: 1.0843125074879807 and parameters: {'num_leaves': 85, 'max_depth': 13, 'learning_rate': 0.01991676509687057, 'n_estimators': 857, 'subsample': 0.9707944623064335, 'colsample_bytree': 0.6742042507311495, 'reg_alpha': 8.694545298433422, 'reg_lambda': 1.1375620575935204, 'min_child_samples': 31, 'subsample_freq': 1, 'scale_pos_weight': 3.0379642815705465}. Best is trial 59 with value: 1.0981483846548816.


Best trial: 59. Best value: 1.09815:  99%|█████████▉| 99/100 [2:03:56<02:02, 122.20s/it]

[I 2025-10-11 23:36:26,921] Trial 98 finished with value: 1.0974503109128195 and parameters: {'num_leaves': 71, 'max_depth': 15, 'learning_rate': 0.013549613519392654, 'n_estimators': 980, 'subsample': 0.9240499663157566, 'colsample_bytree': 0.7188804396488258, 'reg_alpha': 9.79530438480736, 'reg_lambda': 0.7987286224310484, 'min_child_samples': 35, 'subsample_freq': 5, 'scale_pos_weight': 3.967800172630718}. Best is trial 59 with value: 1.0981483846548816.


Best trial: 59. Best value: 1.09815: 100%|██████████| 100/100 [2:06:43<00:00, 76.04s/it] 

[I 2025-10-11 23:39:14,392] Trial 99 finished with value: 1.0939325717716264 and parameters: {'num_leaves': 140, 'max_depth': 15, 'learning_rate': 0.012931336239542169, 'n_estimators': 877, 'subsample': 0.9232293713348572, 'colsample_bytree': 0.7683213271906749, 'reg_alpha': 9.828719852111266, 'reg_lambda': 1.9018985398441577, 'min_child_samples': 36, 'subsample_freq': 5, 'scale_pos_weight': 4.683944244297534}. Best is trial 59 with value: 1.0981483846548816.
Best trial score: 1.0981483846548816
Best params:
  num_leaves: 65
  max_depth: 15
  learning_rate: 0.01934094693360293
  n_estimators: 946
  subsample: 0.8016744086728401
  colsample_bytree: 0.6126077591174907
  reg_alpha: 9.285573815209705
  reg_lambda: 0.5007355411505623
  min_child_samples: 17
  subsample_freq: 2
  scale_pos_weight: 3.7677211288730463


In [96]:
y_pred_proba

array([[4.34782279e-01, 5.65217721e-01],
       [7.84299802e-01, 2.15700198e-01],
       [2.96969380e-01, 7.03030620e-01],
       ...,
       [2.61691209e-04, 9.99738309e-01],
       [2.63614545e-03, 9.97363855e-01],
       [1.46192707e-03, 9.98538073e-01]], shape=(228834, 2))

In [98]:
smote = SMOTE(random_state=42)
X, y = smote.fit_resample(X, y)

# En iyi parametreler
best_params = study.best_trial.params.copy()
best_params['objective'] = 'binary'
best_params['random_state'] = 42

# Final model
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# Tahmin
y_pred_proba = final_model.predict_proba(X)
# Tüm metrikleri yazdır
gini = convert_auc_to_gini(roc_auc_score(y, y_pred_proba[:, 1]))
recall_10 = recall_at_k(y, y_pred_proba, k=0.1)
lift_10 = lift_at_k(y, y_pred_proba, k=0.1)
final_score = ing_hubs_datathon_metric(y, y_pred_proba[:, 1])

print("\n📊 FINAL MODEL SKORLARI (TÜM VERİ ÜZERİNDE):")
print(f"Gini:              {gini:.4f}")
print(f"Recall@10%:        {recall_10:.4f}")
print(f"Lift@10%:          {lift_10:.4f}")
print(f"Final ING Metric:  {final_score:.4f}")


📊 FINAL MODEL SKORLARI (TÜM VERİ ÜZERİNDE):
Gini:              0.9417
Recall@10%:        0.0000
Lift@10%:          0.0000
Final ING Metric:  1.6277


In [33]:
cv_scores = []
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**best_params)
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    score = ing_hubs_datathon_metric(y_val, y_pred_proba)
    cv_scores.append(score)

print(f"\n✅ 5-Fold CV ING Metric: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")


✅ 5-Fold CV ING Metric: 1.1872 ± 0.0201


In [99]:
final_model = lgb.LGBMClassifier(**best_params)

final_model.fit(X, y)

,boosting_type,'gbdt'
,num_leaves,65
,max_depth,15
,learning_rate,0.01934094693360293
,n_estimators,946
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,17


In [100]:
sample_submission["churn"] = final_model.predict(test_data)

In [101]:
sample_submission["churn"].value_counts()

churn
0    33391
1     9615
Name: count, dtype: int64

In [102]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='lgbm with Optuna kfold and feature engineering history data', 
    competition='ing-hubs-turkiye-datathon'
)

100%|██████████| 355k/355k [00:01<00:00, 228kB/s]  


{"message": "Successfully submitted to ING Hubs T\u00fcrkiye Datathon", "ref": 47334336}

In [68]:
sample_submission["churn"].value_counts()

churn
0    23161
1    19845
Name: count, dtype: int64